<a href="https://colab.research.google.com/github/robertbarcik/genai-in-python-tutorial/blob/main/8_structured_outputs/8_structured_outputs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Structured Outputs: JSON You Can Trust

Your helpdesk gets tickets as emails. Your code wants objects: a name, a department, an urgency level, nothing else. Asking the model "please answer in JSON" gets you *something* JSON-shaped, most of the time. **Structured outputs** turn "most of the time" into "every time": you hand the API the exact shape, and the answer is guaranteed to match it.

You already used this once in the basic examples. Here we look under the hood and go through the shapes a real app needs.

## Setup

In [1]:
%pip install -q openai==3.13.0 pydantic==2.12.3 pandas   # Colab installs here; locally, `pip install -r requirements.txt` already covers it

import os
from openai import OpenAI

# Your key, looked up in this order: Colab secret -> environment variable -> a prompt.
try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    from getpass import getpass
    api_key = getpass("OpenAI API key: ")

client = OpenAI(api_key=api_key)
MODEL = "gpt-5.6-luna"   # small, cheap model of the current generation (~$0.20 in / $1.20 out per 1M tokens)

import json
print("Ready. Model:", MODEL)

Note: you may need to restart the kernel to use updated packages.
Ready. Model: gpt-5.6-luna


## Why "answer in JSON" is not enough

Let's do it the naive way three times and compare what comes back. The same email, the same request, no schema.

In [2]:
email = """Hi IT, Jennifer Martinez from Sales here. Since this morning Outlook says my password is wrong
even though I haven't changed it. Client call in an hour, please help ASAP!"""

for attempt in range(3):
    raw = client.responses.create(
        model=MODEL,
        input=f"Extract the support ticket details from this email as JSON:\n{email}",
    ).output_text
    try:
        keys = sorted(json.loads(raw.strip().strip("`").removeprefix("json")).keys())
        print(f"attempt {attempt + 1}: parsed OK, keys = {keys}")
    except json.JSONDecodeError:
        print(f"attempt {attempt + 1}: could not parse, got: {raw[:60]!r}...")

attempt 1: parsed OK, keys = ['issue', 'reason_for_urgency', 'requested_action', 'requester', 'software', 'started', 'urgency']
attempt 2: parsed OK, keys = ['deadline', 'department', 'issue', 'password_changed_by_user', 'reported_start', 'requested_action', 'requester_name', 'urgency']
attempt 3: parsed OK, keys = ['business_impact', 'department', 'issue', 'reported_start', 'requested_action', 'requester_name', 'urgency']


### 🔍 What just happened?

Maybe the three answers matched, maybe one used `name` where another used `user_name`, maybe one wrapped the JSON in Markdown fences. The point is that nothing *enforces* it: the model picked the field names, the types, and the wrapping on its own, and your parsing code has to guess along. That is not something you can build a database import on.

## Describe the shape, get the object

A **Pydantic model** is a Python class that lists fields and their types. Give it to `responses.parse` as `text_format`, and `output_parsed` comes back as an instance of that class, already validated. No parsing, no guessing.

In [3]:
from pydantic import BaseModel


class SupportTicket(BaseModel):
    user_name: str
    department: str
    issue_summary: str
    urgent: bool


ticket = client.responses.parse(
    model=MODEL,
    input=f"Extract the support ticket details from this email:\n{email}",
    text_format=SupportTicket,
).output_parsed

print(ticket)
print("\nurgent is a real boolean:", type(ticket.urgent).__name__, "->", "page on-call!" if ticket.urgent else "queue it")

user_name='Jennifer Martinez' department='Sales' issue_summary='Outlook reports that the password is incorrect even though it has not been changed.' urgent=True

urgent is a real boolean: bool -> page on-call!


### 🔍 What just happened?

Run the cell ten times and you get the same four fields with the same types every time. The model did not "decide" to comply; the API constrained what it was allowed to write, token by token, to text that fits the schema.

### 🎯 Mini-task

Add `category: str` to `SupportTicket` and run again. Then rename `urgent` to `needs_callback` and see how the model interprets the new name.

## What is actually sent

Pydantic is a convenience. Under the hood the API receives a **JSON Schema**: a standard way of writing "an object with these fields of these types". Here is the schema your class turns into, and the raw call without Pydantic, so you recognise it in other languages and in the docs.

In [4]:
print(json.dumps(SupportTicket.model_json_schema(), indent=2))

{
  "properties": {
    "user_name": {
      "title": "User Name",
      "type": "string"
    },
    "department": {
      "title": "Department",
      "type": "string"
    },
    "issue_summary": {
      "title": "Issue Summary",
      "type": "string"
    },
    "urgent": {
      "title": "Urgent",
      "type": "boolean"
    }
  },
  "required": [
    "user_name",
    "department",
    "issue_summary",
    "urgent"
  ],
  "title": "SupportTicket",
  "type": "object"
}


In [5]:
schema = SupportTicket.model_json_schema()
schema["additionalProperties"] = False          # strict mode needs this: no fields beyond the listed ones

response = client.responses.create(
    model=MODEL,
    input=f"Extract the support ticket details from this email:\n{email}",
    text={"format": {"type": "json_schema", "name": "support_ticket", "schema": schema, "strict": True}},
)

print(response.output_text)                      # a JSON string that matches the schema
print(json.loads(response.output_text)["department"])

{"user_name":"Jennifer Martinez","department":"Sales","issue_summary":"Outlook reports that the password is incorrect, although it has not been changed. Assistance is needed before a client call in one hour.","urgent":true}
Sales


**Decode of the key line:** `text={"format": {"type": "json_schema", ..., "strict": True}}` is the whole feature. `strict: True` is what turns "try to follow this" into "must follow this"; it requires every field to be listed in `required` and `additionalProperties: False`. `responses.parse` writes all of that for you from the Pydantic class, which is why we use it everywhere else.

## The shapes a real app needs

Three things come up in every extraction job: a field that may only take certain values, a field that may be missing, and a list of things inside the thing. Pydantic has one idiom for each; the API honours all three.

In [6]:
from typing import Literal, Optional


class Step(BaseModel):
    order: int
    action: str


class Ticket(BaseModel):
    user_name: str
    department: Literal["Sales", "Engineering", "Finance", "HR", "Other"]   # a fixed set of choices
    urgency: Literal["low", "medium", "high", "critical"]
    callback_phone: Optional[str]                                          # may be null when not mentioned
    suggested_steps: list[Step]                                            # a list of nested objects


messy = """Subject: PRODUCTION DOWN - payment gateway
This is Priya Nair, Finance. Since 09:40 none of the card payments on the webshop go through,
customers get "gateway timeout". We are losing orders every minute. Call me on 555-0142."""

ticket = client.responses.parse(
    model=MODEL,
    input=f"Extract the ticket and propose 3 first steps for the technician:\n{messy}",
    text_format=Ticket,
).output_parsed

print(ticket.model_dump_json(indent=2))

{
  "user_name": "Priya Nair",
  "department": "Finance",
  "urgency": "critical",
  "callback_phone": "555-0142",
  "suggested_steps": [
    {
      "order": 1,
      "action": "Declare a production incident and immediately check the payment gateway provider status, webshop payment-service health, and recent deployment or configuration changes."
    },
    {
      "order": 2,
      "action": "Review application, gateway, and network logs for timeout patterns; test a transaction in a controlled manner and verify DNS, TLS, firewall, and API connectivity to the gateway."
    },
    {
      "order": 3,
      "action": "Escalate to the payment gateway provider and on-call engineering team, while informing Priya and business stakeholders of the incident status and any temporary checkout or payment-processing workaround."
    }
  ]
}


### 🔍 What just happened?

`department` could only be one of five strings, so the model could not answer "Accounting". `callback_phone` was filled because the email had a number; run the same call on the first email and it comes back `null`, not a made-up number. `suggested_steps` is a list of real `Step` objects: `ticket.suggested_steps[0].action` just works.

In [7]:
first_ticket = client.responses.parse(
    model=MODEL,
    input=f"Extract the ticket and propose 3 first steps for the technician:\n{email}",
    text_format=Ticket,
).output_parsed

print("phone in the first email:", first_ticket.callback_phone)
print("first step:", first_ticket.suggested_steps[0].action)

phone in the first email: None
first step: Verify whether Jennifer can sign in to the organization’s webmail or other Microsoft 365 services, and confirm the exact Outlook error message.


## When the model refuses

A schema forces the *shape*, not the *willingness*. If a request is unsafe, the model may refuse instead of filling the schema; then `output_parsed` is `None` and the answer carries a `refusal` item. Production code checks for it. Here is the pattern on a normal request (it parses), so you have it ready for the abnormal one.

In [8]:
def extract_ticket(text):
    response = client.responses.parse(
        model=MODEL,
        input=f"Extract the ticket and propose 3 first steps for the technician:\n{text}",
        text_format=Ticket,
    )
    if response.output_parsed is None:                       # no JSON came back
        refusal = next((c.refusal for item in response.output if item.type == "message"
                        for c in item.content if c.type == "refusal"), "no refusal text")
        return None, refusal
    return response.output_parsed, None


parsed, why_not = extract_ticket(messy)
print("parsed:", parsed.user_name if parsed else None, "| refusal:", why_not)

parsed: Priya Nair | refusal: None


## The same machinery powers function calling

A tool's `parameters` block is a JSON Schema too, and `"strict": True` there means the model cannot invent an argument. If you already have a Pydantic model, you can hand its schema straight to a tool definition.

In [9]:
class CreateTicketArgs(BaseModel):
    user_name: str
    urgency: Literal["low", "medium", "high", "critical"]
    issue_summary: str


parameters = CreateTicketArgs.model_json_schema()
parameters["additionalProperties"] = False

tools = [{
    "type": "function",
    "name": "create_ticket",
    "description": "Create a helpdesk ticket in the tracking system.",
    "parameters": parameters,
    "strict": True,
}]

response = client.responses.create(model=MODEL, input=f"File a ticket for this:\n{messy}", tools=tools)
call = next(item for item in response.output if item.type == "function_call")
print(call.name, "->", CreateTicketArgs.model_validate_json(call.arguments))

create_ticket -> user_name='Priya Nair' urgency='critical' issue_summary='PRODUCTION DOWN - payment gateway. Priya Nair (Finance) reports that since 09:40, none of the webshop\'s card payments have gone through; customers receive a "gateway timeout" error. Orders are being lost every minute. Callback: 555-0142.'


### 🔍 What just happened?

The model produced a function call whose arguments validate against `CreateTicketArgs` on the first try. Structured outputs and function calling are two doors into the same room: one returns the object to you, the other hands it to a function.

## 🎯 Your turn

Extract book information from the text below into a `Book` model with `title: str`, `author: str`, `year_published: int` and `genres: list[str]`. Then make `year_published` optional and try a sentence with no year.

```
The book "1984" was written by George Orwell and published in 1949.
It's a dystopian novel that also falls under political fiction and science fiction.
```

A solution is in the next cell; write yours first.

In [10]:
class Book(BaseModel):
    title: str
    author: str
    year_published: Optional[int]
    genres: list[str]


text = ("The book \"1984\" was written by George Orwell and published in 1949. "
        "It's a dystopian novel that also falls under political fiction and science fiction.")

book = client.responses.parse(model=MODEL, input=f"Extract the book details:\n{text}", text_format=Book).output_parsed
print(book)

no_year = client.responses.parse(model=MODEL, input="Extract the book details: Dune by Frank Herbert, a science fiction classic.",
                                 text_format=Book).output_parsed
print(no_year)

title='1984' author='George Orwell' year_published=1949 genres=['Dystopian novel', 'Political fiction', 'Science fiction']
title='Dune' author='Frank Herbert' year_published=None genres=['Science Fiction']


## Where this leads

Every extraction, classification and routing job in the rest of the course uses this one idiom: a Pydantic class and `responses.parse`. Docs: [developers.openai.com/api/docs/guides/structured-outputs](https://developers.openai.com/api/docs/guides/structured-outputs).